In [ ]:
#| default_exp kernel

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory

Run one local or gateway kernel. Execution returns nbformat outputs. Inspection uses the Dhrishti HTTP API.

In [ ]:
#| export
from __future__ import annotations
import asyncio, importlib.util, json, os, queue, shutil, sys, tempfile, time, uuid, warnings
from pathlib import Path
from fastcore.all import L, first, ifnone, store_attr
from fastcore.ansi import strip_ansi
from jupyter_client.manager import AsyncKernelManager
from dhrishti.registry import active, reg_dir
from kunda.pythons import app_name
from kunda.support import HOST_PY, clean_env, strip_bundle, support_paths, work_dir
from kunda.spec import ExecOutcome, KernelStartError, _kernel_env, _runtime_python, _spec, bootstrap_src, kernelspec_for, missing_kernel_module, output_text

`KERNELS` lists supported Python launchers. Unknown launcher names use `ipykernel`.

In [ ]:
#| export
KERNELS, SHUTDOWN_WAIT = ('ipykernel', 'ipymini'), 5.0

In [ ]:
#| export
async def _give_up_after(coro, timeout=SHUTDOWN_WAIT):
    "Await `coro`, then stop waiting rather than hold the caller open. Failing to close is not news."
    try: await asyncio.wait_for(coro, timeout=timeout)
    except Exception: pass

`_give_up_after` stops waiting for shutdown after the timeout.

In [ ]:
#| exec_doc
t = time.time()
await _give_up_after(asyncio.sleep(30), timeout=.05)      # the deadline ends this, not the sleep
async def never_answers(): raise RuntimeError('the kernel never answered')
await _give_up_after(never_answers()), time.time() - t < 1

In [ ]:
#| export
def ipymini_available():
    "True when the ipymini package can be imported in this interpreter."
    return importlib.util.find_spec('ipymini') is not None

`ipymini_available` checks whether the package can be imported without importing it.

In [ ]:
#| exec_doc
ipymini_available()

In [ ]:
#| export
class _Inspector:
    "The inspector-client half shared by `Kernel` and `GatewayKernel`."
    last_used = 0.
    def touch(self):
        "Stamp activity. The idle sweep measures from here, so anything that uses a kernel says so."
        self.last_used = time.time()
        return self
    @property
    def busy(self):
        "A cell is running on this kernel. The sweep never takes one of these."
        t = getattr(self, '_exec_task', None)
        return t is not None and not t.done()
    @property
    def idle_for(self):
        "Seconds since this kernel was last used; 0 while it is running something."
        return 0. if self.busy else max(0., time.time() - self.last_used)
    @property
    def kernel_kind(self):
        "Which launcher this kernel was started with: 'ipykernel' or 'ipymini'."
        return self.kernel
    @property
    def token(self):
        "Owner token for this kernel's inspector, so the IDE may use its gated endpoints."
        if not self.base: return None
        try: return (reg_dir()/f"token-{self.base.rsplit(':', 1)[-1]}").read_text().strip() or None
        except Exception: return None
    def _headers(self): return {'X-Dhrishti-Token': t} if (t := self.token) else {}
    @property
    def _boot_name(self):
        "The name the in-kernel inspector registers itself under."
        return self.name
    async def bootstrap(self):
        "Start the inspector inside the kernel and record its base URL."
        src = bootstrap_src(self._boot_name, self.port, self.agent, True)
        r = await self.execute(src, silent=True, store_history=False)
        if not r.ok:
            self.boot_error = r.error or r.text or 'inspector bootstrap failed'
            return None
        self.base = await self._find_base()
        if self.base is None: self.boot_error = 'inspector started but did not register'
        return self.base
    async def api(self, path, params=None, timeout=15):
        "GET this kernel's own inspector, carrying its owner token; None if it has none."
        if not self.base: return None
        import httpx2 as httpx
        try:
            async with httpx.AsyncClient(base_url=self.base, timeout=timeout, headers=self._headers()) as c:
                r = await c.get(path, params=params or {})
                r.raise_for_status()
                self.last_api_error = ''
                return r.json()
        except Exception as e:
            self.last_api_error = f'{type(e).__name__}: {e}'
            return None
    def api_sync(self, path, params=None, timeout=30):
        "Synchronous inspector call for model tools running on an agent worker thread."
        if not self.base: return None
        import httpx2 as httpx
        try:
            with httpx.Client(base_url=self.base, timeout=timeout, headers=self._headers()) as c:
                r = c.get(path, params=params or {})
                r.raise_for_status()
                self.last_api_error = ''
                return r.json()
        except Exception as e:
            self.last_api_error = f'{type(e).__name__}: {e}'
            return None
    async def names(self, scope='live', profile='full'):
        "Every top-level name in this kernel's namespace, or the agent sandbox's."
        base = '/agent/api/rows' if scope == 'isolated' else '/api/rows'
        r = await self.api(base, {'profile': profile, 'sort': 'name'})
        return L(n.get('name', '') for g in (r or {}).get('groups', []) for n in g.get('nodes', []))
    async def stop(self, timeout=2.0):
        "Interrupt the active execution, then restart the kernel if it ignores the deadline."
        task = self._exec_task
        if task is None or task.done(): return {'ok': False, 'forced': False, 'note': 'nothing is running'}
        await self.interrupt()
        try:
            await asyncio.wait_for(asyncio.shield(task), timeout=timeout)
            return {'ok': True, 'forced': False, 'note': 'cell interrupted'}
        except asyncio.TimeoutError:
            task.cancel()
            await asyncio.gather(task, return_exceptions=True)
            await self.restart()
            return {'ok': True, 'forced': True, 'note': 'kernel restarted after interrupt was ignored'}
        except asyncio.CancelledError: return {'ok': True, 'forced': False, 'note': 'cell stopped'}

`_Inspector` holds state and Dhrishti client methods shared by local and gateway kernels.

In [ ]:
#| hide
#| exec_doc
class Probe(_Inspector):
    "An `_Inspector` with no kernel under it: the parts that need no process."
    _exec_task, kernel = None, 'ipykernel'
    def __init__(self, base=None, name='nb'): self.base, self.name = base, name

In [ ]:
#| hide
#| exec_doc
reg = TemporaryDirectory(); os.environ['DHRISHTI_REG_DIR'] = reg.name
(Path(reg.name)/'token-8123').write_text('S3CRET\n')

In [ ]:
#| exec_doc
p = Probe('http://127.0.0.1:8123')
p.token, p._headers(), Probe().token

In [ ]:
#| exec_doc
p.touch()
p.busy, p.idle_for < 1, p.kernel_kind

`names` returns top-level names from Dhrishti's grouped response.

In [ ]:
#| exec_doc
class Rows(Probe):
    "The inspector's answer, without the inspector."
    async def api(self, path, params=None, timeout=15):
        self.asked = path
        return {'groups': [{'name': 'data', 'nodes': [{'name': 'df'}, {'name': 'raw'}]}, {'name': 'other', 'nodes': [{'name': 'x'}]}]}
r = Rows('http://127.0.0.1:8123')
await r.names(), r.asked

In [ ]:
#| hide
test_eq(await r.names(scope='isolated'), ['df', 'raw', 'x'])
test_eq(r.asked, '/agent/api/rows')          # the sandbox is a second namespace, not a second kernel
test_eq(await Probe().names(), [])
test_is(await Probe().api('/api/rows'), None)
test_is(Probe().api_sync('/api/rows'), None)

`stop` interrupts the active execution. It restarts the kernel when the execution does not stop before the timeout.

In [ ]:
#| hide
#| exec_doc
class Running(Probe):
    "A kernel with a cell on it, which ends when interrupted only if `obeys`."
    def __init__(self, obeys=True):
        super().__init__()
        self.obeys, self.restarted, self.ended = obeys, False, asyncio.Event()
        self._exec_task = asyncio.ensure_future(self.ended.wait())
    async def interrupt(self):
        if self.obeys: self.ended.set()
    async def restart(self): self.restarted = True; self.ended.set()

In [ ]:
#| exec_doc
async def one_cell(obeys):
    k = Running(obeys)
    return (await k.stop(timeout=.05))['note'], k.restarted
print(await Probe().stop())
for obeys in (True, False): print(await one_cell(obeys))

In [ ]:
#| export
class Kernel(_Inspector):
    "One ipykernel process: execute/complete over jupyter_client, live variables over its in-kernel."
    def __init__(self,
        cwd=None,
        name=None,
        python=None,
        inspect=True,
        agent='restricted',
        port=8000,
        kernel='ipykernel',
        lang='python',
        known=None,        # the host's `{language: kernelspec}`, handed down by the pool
        install='',        # what would install a kernel for `lang`, for the error when none is
    ):
        if kernel not in KERNELS: kernel = 'ipykernel'
        if lang != 'python': inspect, kernel = False, 'ipykernel'
        if kernel == 'ipymini' and not ipymini_available():
            warnings.warn("ipymini is unavailable in this interpreter; falling back to ipykernel. Install it with `install_kernel_support`.")
            kernel = 'ipykernel'
        store_attr()
        self.km = self.kc = None
        self._ipc_dir = None
        self.base = None
        self.boot_error = None
        self.last_api_error = ''
        self._exec_lock = asyncio.Lock()
        self._exec_task = None
        self._shell_lock = asyncio.Lock()
        self._debug_lock = asyncio.Lock()
        self._debug_seq = 0
        self._debug_thread = None
        self._debug_source = None
        self._debug_sources = {}
    @property
    def alive(self):
        "True only once the process and its usable client are both ready."
        return self.km is not None and self.kc is not None and self.km.has_kernel
    @property
    def pid(self):
        "OS pid of the kernel process, or None."
        p = getattr(getattr(self.km, 'provisioner', None), 'process', None)
        return getattr(p, 'pid', None)
    async def chdir(self, path):
        "Move the kernel to `path`. Only Python is asked in its own language; nothing else is asked."
        if self.lang != 'python': return False
        r = await self.execute(f'import os as __leela_os; __leela_os.chdir({str(path)!r}); del __leela_os')
        if not r.ok: raise RuntimeError(r.error or f'could not change kernel cwd to {path}')
        self.cwd = path
        return True
    async def start(self, timeout=60):
        "Launch the kernel, wait for it to answer, then bootstrap its inspector."
        spec_name = kernelspec_for(self.lang, self.known) if self.lang != 'python' else None
        manager_kw = {'kernel_name': spec_name or (self.kernel if self.kernel == 'ipymini' else 'python3')}
        if os.name != 'nt' and self.kernel == 'ipykernel' and self.lang == 'python':
            self._ipc_dir = tempfile.mkdtemp(prefix='lee-k-', dir='/tmp' if os.path.isdir('/tmp') else None)
            manager_kw.update(transport='ipc', ip=os.path.join(self._ipc_dir, 'k'))
        elif os.name != 'nt': manager_kw.update(transport='tcp', ip='127.0.0.1')
        elif self.kernel == 'ipykernel':
            manager_kw['transport_encryption'] = 'required'
        self.km = AsyncKernelManager(**manager_kw)
        self.km._kernel_spec = _spec(self.python, ifnone(self.name, 'python3'), self.kernel, self.lang, self.known, self.install)
        cwd = work_dir(self.cwd)
        try:
            await self.km.start_kernel(cwd=cwd, env=_kernel_env(self.python))
            client_kw = {}
            if getattr(self.km, 'curve_publickey', None) is not None:
                client_kw.update(curve_publickey=self.km.curve_publickey,curve_secretkey=self.km.curve_secretkey)
            self.kc = self.km.client(**client_kw)
            self.kc.start_channels()
            await self.kc.wait_for_ready(timeout=timeout)
        except BaseException as e:
            await self.shutdown()
            if isinstance(e, Exception) and (mod := missing_kernel_module(self.python, self.kernel)):
                raise KernelStartError(
                    f'{_runtime_python(self.python)} cannot import {mod}, so it can run no kernel. '
                    f'Run `uv sync` in that project, or pick another interpreter in the kernel picker.') from e
            raise
        if self.inspect: await self.bootstrap()
        return self
    async def _find_base(self):
        "The registry entry written by our own kernel process, matched on pid."
        for _ in range(40):
            if (e := first(active(), lambda e: e.get('pid') == self.pid)): return e['base']
            await asyncio.sleep(0.05)
        return None
    async def interrupt(self):
        "Request cooperative cancellation; callers needing a guarantee should use stop()."
        if self.km: await self.km.interrupt_kernel()
    async def restart(self):
        "Restart the process; the namespace and the in-kernel inspector are both rebuilt."
        if not self.km: return
        self.base = self.boot_error = None
        await self.km.restart_kernel(now=False)
        await self.kc.wait_for_ready(timeout=60)
        if self.inspect: await self.bootstrap()
    async def shutdown(self):
        if self.kc:
            try: self.kc.stop_channels()
            except Exception: pass
        if self.km and self.km.has_kernel:
            try: await self.km.shutdown_kernel(now=False)
            except Exception: pass
        self.km = self.kc = self.base = None
        if self._ipc_dir:
            shutil.rmtree(self._ipc_dir, ignore_errors=True)
            self._ipc_dir = None
    async def execute(self, code, silent=False, store_history=True, on_output=None, timeout=None,
        on_debug=None):
        "Run `code` on the shell channel and collect nbformat-shaped outputs, streaming each to `on_output`."
        if not self.alive: return ExecOutcome(ok=False, error='kernel is not running')
        async with self._exec_lock, self._shell_lock:
            if not self.alive: return ExecOutcome(ok=False, error='kernel is not running')
            task = asyncio.current_task()
            self._exec_task = task
            self.touch()
            try:
                msg_id = self.kc.execute(code, silent=silent, store_history=store_history,
                    allow_stdin=False)
                outs = await self._collect(msg_id, on_output, timeout, on_debug)
                reply = await self._reply(msg_id, timeout)
            finally:
                self.touch()
                if self._exec_task is task: self._exec_task = None
        c = reply.get('content', {}) if reply else {}
        err = None
        if c.get('status') == 'error': err = f"{c.get('ename')}: {c.get('evalue')}"
        elif c.get('status') == 'abort': err = 'aborted'
        if err is None and (e := first(outs, lambda o: o['output_type'] == 'error')): err = f"{e.get('ename')}: {e.get('evalue')}"
        return ExecOutcome(ok=err is None, execution_count=c.get('execution_count'), outputs=outs, error=err)
    async def _collect(self, msg_id, on_output=None, timeout=None, on_debug=None):
        "Drain iopub for one request, applying Jupyter display-id updates in place."
        outs, displays = [], {}
        while True:
            try: msg = await self.kc.get_iopub_msg(timeout=timeout or 0.5)
            except (queue.Empty, asyncio.TimeoutError):
                if not self.alive: break
                continue
            t, c = msg['header']['msg_type'], msg['content']
            if t == 'debug_event':
                if c.get('event') == 'stopped': self._debug_thread = (c.get('body') or {}).get('threadId')
                elif c.get('event') == 'continued': self._debug_thread = None
                if on_debug: on_debug(c)
                continue
            if (msg.get('parent_header') or {}).get('msg_id') != msg_id: continue
            if t == 'status':
                if c.get('execution_state') == 'idle': break
            elif t == 'clear_output': outs.clear()
            elif o := self._as_output(t, c):
                display_id = (c.get('transient') or {}).get('display_id')
                if t == 'update_display_data' and display_id in displays:
                    outs[displays[display_id]] = o
                elif (o['output_type'] == 'stream' and outs and outs[-1].get('output_type') == 'stream'
                    and outs[-1].get('name') == o['name']): outs[-1]['text'] += o['text']
                else:
                    outs.append(o)
                    if display_id: displays[display_id] = len(outs) - 1
                if on_output:
                    event = dict(o)
                    if display_id: event.update(_display_id=display_id, _update=(t == 'update_display_data'))
                    on_output(event)
        return outs
    @staticmethod
    def _as_output(msg_type, c):
        "An iopub message as an nbformat output dict, or None if it carries no output."
        if msg_type == 'stream': return {'output_type': 'stream', 'name': c.get('name', 'stdout'), 'text': c.get('text', '')}
        if msg_type == 'error':
            return {'output_type': 'error', 'ename': c.get('ename', ''), 'evalue': c.get('evalue', ''),
                'traceback': list(c.get('traceback') or [])}
        if msg_type in ('execute_result', 'display_data', 'update_display_data'):
            o = {'output_type': 'execute_result' if msg_type == 'execute_result' else 'display_data',
                'data': c.get('data') or {}, 'metadata': c.get('metadata') or {}}
            if msg_type == 'execute_result': o['execution_count'] = c.get('execution_count')
            return o
        return None
    async def _reply(self, msg_id, timeout=None):
        "The shell reply for one request, skipping replies to anything else."
        while True:
            try: msg = await self.kc.get_shell_msg(timeout=timeout or 10)
            except (queue.Empty, asyncio.TimeoutError): return None
            if (msg.get('parent_header') or {}).get('msg_id') == msg_id: return msg
    async def debug_request(self, command, arguments=None, timeout=15):
        "Send one Jupyter Debug Protocol/DAP request over ipykernel's control channel."
        if self.kernel != 'ipykernel': raise RuntimeError('cell debugging currently requires an ipykernel runtime')
        if not self.alive: raise RuntimeError('kernel is not running')
        async with self._debug_lock:
            self._debug_seq += 1
            content = {'seq': self._debug_seq, 'type': 'request', 'command': command,
                'arguments': arguments or {}}
            msg = self.kc.session.msg('debug_request', content=content)
            self.kc.control_channel.send(msg)
            while True:
                try: reply = await self.kc.get_control_msg(timeout=timeout)
                except (queue.Empty, asyncio.TimeoutError):
                    raise TimeoutError(f'debugger did not answer {command}')
                if (reply.get('parent_header') or {}).get('msg_id') != msg['header']['msg_id']: continue
                out = reply.get('content') or {}
                if not out.get('success', False): raise RuntimeError(out.get('message') or f'debugger rejected {command}')
                return out
    async def debug_start(self, code, breakpoints=(), notebook_cells=()):
        "Initialize debugpy and map every notebook cell source to ipykernel's debug files."
        await self.debug_request('initialize', {
            'clientID': 'leela', 'clientName': 'Leela', 'adapterID': 'python',
            'pathFormat': 'path', 'linesStartAt1': True, 'columnsStartAt1': True,
            'supportsVariableType': True, 'supportsVariablePaging': True,
        })
        await self.debug_request('attach')
        self._debug_sources = {}
        for cell_id, source in notebook_cells:
            dumped = await self.debug_request('dumpCell', {'code': source})
            if (path := dumped.get('body', {}).get('sourcePath')): self._debug_sources[path] = cell_id
        dumped = await self.debug_request('dumpCell', {'code': code})
        self._debug_source = dumped.get('body', {}).get('sourcePath')
        if not self._debug_source: raise RuntimeError('ipykernel did not create a debug source for the cell')
        requested = [{'line': max(1, int(line))} for line in breakpoints]
        set_reply = await self.debug_request('setBreakpoints', {
            'source': {'path': self._debug_source}, 'breakpoints': requested,
            'sourceModified': False,
        })
        await self.debug_request('configurationDone')
        return [b.get('line') for b in set_reply.get('body', {}).get('breakpoints', [])
            if b.get('verified', True)]
    async def debug_variables(self, reference, start=0, count=200):
        "Children of one paused DAP variable, preserving references for recursive expansion."
        reply = await self.debug_request('variables', {
            'variablesReference': int(reference), 'start': int(start), 'count': int(count),
        })
        return reply.get('body', {}).get('variables', [])
    async def debug_state(self, thread_id=None, frame_id=None):
        "The paused call stack and locals for the selected frame, normalized for the browser."
        thread_id = int(thread_id or self._debug_thread or 0)
        if not thread_id: return {'paused': False, 'frames': [], 'variables': []}
        stack = await self.debug_request('stackTrace', {'threadId': thread_id,
            'startFrame': 0, 'levels': 50})
        frames = stack.get('body', {}).get('stackFrames', [])
        selected = first(frames, lambda f: f.get('id') == int(frame_id or -1)) or first(frames)
        variables, scopes = [], []
        if selected:
            scope_reply = await self.debug_request('scopes', {'frameId': selected['id']})
            scopes = scope_reply.get('body', {}).get('scopes', [])
            local = first(scopes, lambda s: s.get('name', '').lower() == 'locals') or first(scopes)
            if local and local.get('variablesReference'): variables = await self.debug_variables(local['variablesReference'])
        return {'paused': True, 'thread_id': thread_id, 'frame_id': selected.get('id') if selected else None,
            'frames': frames, 'scopes': scopes, 'variables': variables,
            'source': self._debug_source}
    async def debug_evaluate(self, expression, frame_id=None):
        "Evaluate an expression in the selected paused frame, the debugger's exec console."
        if not self._debug_thread: raise RuntimeError('the debugger is not paused')
        if frame_id is None:
            state = await self.debug_state()
            frame_id = state.get('frame_id')
        reply = await self.debug_request('evaluate', {
            'expression': str(expression), 'frameId': int(frame_id), 'context': 'repl',
        })
        return reply.get('body', {})
    async def debug_complete(self, code, pos, frame_id=None):
        "Completions from the paused frame without queueing a shell request behind it."
        if frame_id is None:
            state = await self.debug_state()
            frame_id = state.get('frame_id')
        reply = await self.debug_request('completions', {
            'text': str(code), 'column': int(pos) + 1, 'frameId': int(frame_id),
        })
        targets = reply.get('body', {}).get('targets', [])
        return {'from': int(pos), 'matches': [t.get('text') or t.get('label') for t in targets
            if t.get('text') or t.get('label')][:50]}
    async def debug_control(self, command, thread_id=None):
        if command not in {'continue', 'next', 'stepIn', 'stepOut', 'pause'}: raise ValueError(f'unknown debug command {command!r}')
        tid = int(thread_id or self._debug_thread or 0)
        if not tid: raise RuntimeError('the debugger is not paused')
        out = await self.debug_request(command, {'threadId': tid})
        if command != 'pause': self._debug_thread = None
        return out
    async def debug_stop(self):
        if self.kernel != 'ipykernel': return
        try: await self.debug_request('disconnect', {'restart': False, 'terminateDebuggee': False})
        finally:
            self._debug_thread = self._debug_source = None
            self._debug_sources = {}
    async def complete(self, code, pos):
        "Completions at `pos` in the browser's `{from, matches}` response shape."
        if not self.alive: return {'from': pos, 'matches': []}
        async with self._shell_lock:
            msg_id = self.kc.complete(code, pos)
            r = await self._reply(msg_id)
        c = (r or {}).get('content', {})
        return {'from': c.get('cursor_start', pos), 'matches': list(c.get('matches') or [])[:50]}
    async def inspect_obj(self, code, pos, detail=0):
        "Kernel-side introspection (signature and docstring) for the object at `pos`; '' when unknown."
        if not self.alive: return ''
        async with self._shell_lock:
            msg_id = self.kc.inspect(code, pos, detail_level=detail)
            r = await self._reply(msg_id)
        c = (r or {}).get('content', {})
        if not c.get('found'): return ''
        return strip_ansi((c.get('data') or {}).get('text/plain') or '')

`Kernel` owns one local kernel process and its Jupyter client. `start` waits for the kernel and then starts Dhrishti when inspection is enabled.

In [ ]:
#| exec_doc
k = Kernel(kernel='not-a-kernel', inspect=False)
j = Kernel(cwd='/proj', lang='julia', inspect=True)
k.kernel, (j.kernel, j.inspect, await j.chdir('/proj'))

Execution on a stopped kernel returns a failed `ExecOutcome`. Each request collects only messages with its own parent ID.

In [ ]:
#| exec_doc
k.alive, k.pid, (await k.execute('1+1')).error, await k.complete('1+', 2), await k.inspect_obj('1+', 2)

In [ ]:
#| hide
test_eq((await k.execute('1+1')).outputs, [])
with ExceptionExpected(RuntimeError, 'not running'): await k.debug_request('initialize')

Registered language kernels use their installed kernelspec. A missing kernelspec raises `KernelStartError`.

In [ ]:
#| hide
seen, _kernelspec_for = [], kernelspec_for
def kernelspec_for(lang, known=None):
    seen.append((lang, known))
    return None
err = ''
try: await Kernel(lang='rust', known={'rust': 'evcxr'}, install='cargo install evcxr_jupyter').start()
except KernelStartError as e: err = str(e)
finally: kernelspec_for = _kernelspec_for
test_eq(seen, [('rust', {'rust': 'evcxr'})])
test_eq(err, 'no Jupyter kernel is installed for rust. Install one with `cargo install evcxr_jupyter`.')

`_as_output` converts one IOPub message to an nbformat output. Messages without output return `None`.

In [ ]:
#| exec_doc
[Kernel._as_output('stream', {'name': 'stdout', 'text': 'hello\n'}),
 Kernel._as_output('execute_result', {'data': {'text/plain': '42'}, 'execution_count': 1}),
 Kernel._as_output('update_display_data', {'data': {'text/plain': 'second'}}),
 Kernel._as_output('status', {'execution_state': 'idle'})]

In [ ]:
#| hide
test_eq(Kernel._as_output('error', {'ename': 'ValueError', 'evalue': 'nope'}),
        {'output_type': 'error', 'ename': 'ValueError', 'evalue': 'nope', 'traceback': []})
test_eq(Kernel._as_output('display_data', {}), {'output_type': 'display_data', 'data': {}, 'metadata': {}})
test_is(Kernel._as_output('execute_input', {'code': '1+1'}), None)

In [ ]:
#| export
#: What a gateway needs, with the version range each import has to satisfy.
#: `jupyasyncclient` 0.2.8 leaves the gateway executing nothing, so it is excluded here as in `pyproject.toml`.
GATEWAY_DEPS = (('jupygate', ''), ('jupyasyncclient', '>=0.2.1,!=0.2.8'))

In [ ]:
#| export
def check_gateway_deps():
    "Check the gateway dependencies and versions."
    import importlib.metadata
    from packaging.specifiers import SpecifierSet
    for name, spec in GATEWAY_DEPS:
        try: importlib.import_module(name)
        except ImportError as e: raise RuntimeError(
            f'gateway transport needs {name}, which is not installed: '
            f'pip install {name}') from e
        if not spec: continue
        try: version = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: continue
        if not SpecifierSet(spec, prereleases=True).contains(version): raise RuntimeError(
            f'gateway transport needs {name}{spec}, and {version} is installed; '
            f'reinstall with `pip install "{name}{spec}"`')

`check_gateway_deps` reports a missing or incompatible gateway package. `jupyasyncclient` 0.2.8 is excluded because it does not execute requests.

In [ ]:
#| exec_doc
try: check_gateway_deps()
except RuntimeError as e: print(e)

In [ ]:
#| export
class GatewayService:
    "One supervised jupygate for all workspace kernels in this host process."
    def __init__(self, port=8787, token=None):
        import secrets
        self.port, self.token, self.server = int(port), token or secrets.token_urlsafe(24), None
    @property
    def url(self): return getattr(self.server, 'url', None) or f'http://127.0.0.1:{self.port}'
    def start(self):
        check_gateway_deps()
        if self.server is not None: return self
        from jupygate.core import create_app, serve
        argv = [sys.executable, '-m', 'ipykernel_launcher', '-f', '{connection_file}']
        try: self.server = serve(create_app(argv=argv, auth_token=self.token), port=self.port, in_thread=True)
        except OSError as e: raise RuntimeError(
            f'gateway transport could not listen on port {self.port}; another {app_name()} or a '
            'leftover gateway is holding it') from e
        return self
    def stop(self):
        if self.server is not None: self.server.should_exit = True

`GatewayService` starts one Jupyter gateway for this process and returns its connection details.

In [ ]:
#| exec_doc
g = GatewayService(port='8899', token='shared-secret')
g.url, g.server

In [ ]:
#| hide
test_eq(GatewayService(port='8899').port, 8899)
assert GatewayService().token != GatewayService().token, 'a generated token belongs to one service'

In [ ]:
#| export
class GatewayKernel(_Inspector):
    "The `Kernel` interface over jupygate + jupyasyncclient, so a kernel outlives a client reconnect."
    lang = 'python'
    def __init__(self, gateway, cwd=None, name=None, python=None, inspect=True,
        agent='restricted', port=8000, kernel='ipykernel'):
        store_attr()
        if self.kernel not in KERNELS: self.kernel = 'ipykernel'
        if self.kernel == 'ipymini' and not ipymini_available():
            warnings.warn('ipymini not installed; falling back to ipykernel')
            self.kernel = 'ipykernel'
        self.kc, self.base, self.boot_error, self._alive = None, None, None, False
        self.last_api_error = ''
        self._registry_name = f'{self.name or "kernel"}:{uuid.uuid4().hex[:10]}'
        self._exec_lock = asyncio.Lock()
        self._exec_task = None
    @property
    def alive(self): return self._alive and self.kc is not None
    @property
    def pid(self): return None
    async def start(self, timeout=60):
        check_gateway_deps()
        from jupyasyncclient import JupyAsyncKernelClient
        self.gateway.start()
        argv = _spec(self.python, ifnone(self.name, 'python3'), self.kernel).argv
        self.kc = await JupyAsyncKernelClient.connect(self.gateway.url, token=self.gateway.token,
            timeout=timeout, argv=argv,
            env=_kernel_env(self.python),
            cwd=work_dir(self.cwd))
        self._alive = True
        if self.inspect: await self.bootstrap()
        return self
    @property
    def _boot_name(self):
        "jupygate exposes no pid, so the registry entry is matched on a name unique to this kernel."
        return self._registry_name
    async def _find_base(self):
        "The registry entry written by this kernel, matched on its unique name."
        for _ in range(60):
            if rows := [e for e in active() if e.get('name') == self._registry_name]: return rows[-1]['base']
            await asyncio.sleep(.05)
        return None
    async def execute(self, code, silent=False, store_history=True, on_output=None, timeout=None):
        if not self.alive: return ExecOutcome(ok=False, error='kernel is not running')
        async with self._exec_lock:
            task = asyncio.current_task()
            self._exec_task = task
            self.touch()
            run = self.kc.run(code, silent=silent, store_history=store_history)
            outputs = []
            try:
                async for output in run:
                    outputs.append(output)
                    if on_output: on_output(output)
            except Exception as e: return ExecOutcome(ok=False, outputs=outputs, error=str(e))
            finally:
                self.touch()
                if self._exec_task is task: self._exec_task = None
        reply = (run.reply or {}).get('content', {})
        error = None if run.status == 'ok' else f"{reply.get('ename', run.status)}: {reply.get('evalue', '')}".rstrip(': ')
        return ExecOutcome(ok=error is None, execution_count=run.execution_count,
            outputs=outputs, error=error)
    async def complete(self, code, pos):
        if not self.alive: return {'from': pos, 'matches': []}
        matches, start = await self.kc.complete(code, pos)
        return {'from': start, 'matches': list(matches)[:50]}
    async def inspect_obj(self, code, pos, detail=0):
        if not self.alive: return ''
        return strip_ansi(await self.kc.inspect(code, pos, detail_level=detail))
    async def interrupt(self):
        "Request cooperative cancellation through jupygate's kernel control endpoint."
        if self.kc: await self.kc.interrupt_kernel()
    async def restart(self):
        if not self.kc: return
        self.base = self.boot_error = None
        await self.kc.restart_kernel(); await self.kc.wait_for_ready(timeout=60)
        if self.inspect: await self.bootstrap()
    async def _release(self):
        await self.kc.shutdown_kernel()
        await self.kc.aclose()
    async def shutdown(self):
        if self.kc: await _give_up_after(self._release())
        self.kc, self.base, self._alive = None, None, False

`GatewayKernel` implements the `Kernel` interface over a gateway WebSocket. The gateway owns the kernel process.

In [ ]:
#| exec_doc
gk = GatewayKernel(None, name='nb', inspect=False)
gk.alive, gk.pid, gk.lang, gk._boot_name.startswith('nb:'), (await gk.execute('1+1')).error

In [ ]:
#| hide
assert gk._boot_name != GatewayKernel(None, name='nb')._boot_name, 'one notebook, two kernels, two names'
test_eq(await gk.complete('1+', 2), {'from': 2, 'matches': []})
test_eq(await gk.inspect_obj('1+', 2), '')
test_eq(GatewayKernel(None, kernel='not-a-kernel').kernel, 'ipykernel')

In [ ]:
#| hide
os.environ.pop('DHRISHTI_REG_DIR', None); reg.cleanup()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()